In [1]:
import pandas as pd
import numpy as np

df = pd.read_parquet("../data/processed/sepsis_cleaned_final.parquet")
print(df.shape)

(1552210, 42)


In [2]:
# Get all unique patient IDs from the dataset
patients = df["patient_id"].unique()

# Print total number of unique patients
print(len(patients), "total unique patients")

# Set a random seed so the shuffle is reproducible (same result every time we run this)
np.random.seed(42)

# Shuffle the patient IDs randomly
np.random.shuffle(patients)

# Calculate split points: 70% train, 15% validation, 15% test
n = len(patients)
train_ids = patients[:int(0.7*n)]
val_ids   = patients[int(0.7*n):int(0.85*n)]
test_ids  = patients[int(0.85*n):]

# Print how many patients are in each split
print("Train patients:", len(train_ids))
print("Val patients:", len(val_ids))
print("Test patients:", len(test_ids))

40336 total unique patients
Train patients: 28235
Val patients: 6050
Test patients: 6051


In [3]:
# Select all rows belonging to train patients
train_df = df[df["patient_id"].isin(train_ids)].copy()

# Select all rows belonging to validation patients
val_df = df[df["patient_id"].isin(val_ids)].copy()

# Select all rows belonging to test patients
test_df = df[df["patient_id"].isin(test_ids)].copy()

# Print shapes to confirm the split worked
print("Train rows:", train_df.shape)
print("Val rows:", val_df.shape)
print("Test rows:", test_df.shape)

# Check sepsis percentage in each split - should be roughly similar across all three
print("Train sepsis %:", train_df["SepsisLabel"].mean())
print("Val sepsis %:", val_df["SepsisLabel"].mean())
print("Test sepsis %:", test_df["SepsisLabel"].mean())

Train rows: (1085255, 42)
Val rows: (230301, 42)
Test rows: (236654, 42)
Train sepsis %: 0.01806395731878683
Val sepsis %: 0.01713844056256812
Test sepsis %: 0.01844464915023621


In [4]:
# Function to add rolling window statistics (mean and standard deviation) for given columns
def add_rolling_features(data, cols, windows=[6, 12]):
    # Sort by patient and time, so rolling calculations happen in correct time order
    data = data.sort_values(["patient_id", "ICULOS"])
    
    for col in cols:
        for w in windows:
            # Calculate rolling mean (average) over the past 'w' hours, within each patient
            data[f"{col}_mean{w}"] = data.groupby("patient_id")[col].transform(
                lambda x: x.rolling(w, min_periods=1).mean()
            ).astype("float32")
            
            # Calculate rolling standard deviation (how much values fluctuate) over past 'w' hours
            data[f"{col}_std{w}"] = data.groupby("patient_id")[col].transform(
                lambda x: x.rolling(w, min_periods=1).std()
            ).astype("float32")
    
    return data

# List of vital signs we want rolling features for
vitals = ["HR", "O2Sat", "Temp", "SBP", "MAP", "DBP", "Resp"]

print("function ready")

function ready


In [5]:
# Apply rolling features to the training set
train_df = add_rolling_features(train_df, vitals, windows=[6, 12])
print("Train shape after rolling features:", train_df.shape)

# Apply rolling features to the validation set
val_df = add_rolling_features(val_df, vitals, windows=[6, 12])
print("Val shape after rolling features:", val_df.shape)

# Apply rolling features to the test set
test_df = add_rolling_features(test_df, vitals, windows=[6, 12])
print("Test shape after rolling features:", test_df.shape)

Train shape after rolling features: (1085255, 70)
Val shape after rolling features: (230301, 70)
Test shape after rolling features: (236654, 70)


In [6]:
# Function to calculate clinical scores that doctors use in real hospitals
def add_clinical_scores(data):
    # Shock Index = Heart Rate divided by Systolic Blood Pressure
    # Higher value indicates higher risk of shock
    data["ShockIndex"] = (data["HR"] / data["SBP"].replace(0, 1)).astype("float32")
    
    # qSOFA score: +1 point if respiratory rate is high, +1 point if blood pressure is low
    data["qSOFA"] = ((data["Resp"] >= 22).astype("int8") + 
                      (data["SBP"] <= 100).astype("int8"))
    
    # SIRS score: +1 point each for high heart rate, high respiratory rate, and abnormal temperature
    data["SIRS"] = ((data["HR"] > 90).astype("int8") + 
                     (data["Resp"] > 20).astype("int8") + 
                     ((data["Temp"] > 38) | (data["Temp"] < 36)).astype("int8"))
    
    return data

# Apply this function to all three datasets
train_df = add_clinical_scores(train_df)
val_df = add_clinical_scores(val_df)
test_df = add_clinical_scores(test_df)

# Print a few rows to verify the scores look correct
print(train_df[["HR", "SBP", "Resp", "Temp", "ShockIndex", "qSOFA", "SIRS"]].head(10))

      HR    SBP  Resp       Temp  ShockIndex  qSOFA  SIRS
0   97.0   98.0  19.0  37.065447    0.989796      1     1
1   97.0   98.0  19.0  37.061306    0.989796      1     1
2   89.0  122.0  22.0  37.044090    0.729508      1     1
3   90.0  122.0  30.0  38.534609    0.737705      1     2
4  103.0  122.0  24.5  38.570668    0.844262      1     3
5  110.0  122.0  22.0  37.317670    0.901639      1     2
6  108.0  123.0  29.0  36.110000    0.878049      1     2
7  106.0   93.0  29.0  37.065385    1.139785      2     2
8  104.0  133.0  26.0  37.946929    0.781955      1     2
9  102.0  134.0  30.0  38.093089    0.761194      1     3


In [7]:
# Find all columns that contain "_std" in their name (these are our rolling standard deviation columns)
std_cols = [c for c in train_df.columns if "_std" in c]

# Fill any missing values in these columns with 0 (happens only for the very first hour of each patient)
for d in [train_df, val_df, test_df]:
    d[std_cols] = d[std_cols].fillna(0)

# Final check: confirm no missing values remain anywhere in each dataset
print("Missing in train:", train_df.isna().sum().sum())
print("Missing in val:", val_df.isna().sum().sum())
print("Missing in test:", test_df.isna().sum().sum())

Missing in train: 0
Missing in val: 0
Missing in test: 0


In [8]:
# Save the final, fully processed datasets (cleaned + split + feature engineered)
train_df.to_parquet("../data/processed/train_final.parquet")
val_df.to_parquet("../data/processed/val_final.parquet")
test_df.to_parquet("../data/processed/test_final.parquet")

print("all final datasets saved!")

all final datasets saved!
